# FIFA World Cup 2026 Prediction Model

**Business Question:** Which team is most likely to win the 2026 FIFA World Cup? What are the probabilities for 1st, 2nd, 3rd, and 4th place?

**Approach:**
1. Build a custom Elo rating system from 150+ years of international results
2. Train a match outcome model on Elo rating differences
3. Simulate the full 2026 tournament using the **official group draw** — group stage + knockout — 10,000 times
4. Output probabilities for Champion, Runner-up, 3rd Place, and 4th Place

**Data:** International football results 1872–present ([source](https://github.com/martj42/international_results))

---

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

## 1. Load Data

In [ ]:
url = 'https://raw.githubusercontent.com/martj42/international_results/master/results.csv'
df  = pd.read_csv(url, parse_dates=['date'])

# Drop future scheduled matches (no scores yet)
df = df.dropna(subset=['home_score', 'away_score'])
df['home_score'] = df['home_score'].astype(int)
df['away_score'] = df['away_score'].astype(int)

print(f"Total matches: {len(df):,}")
print(f"Date range:    {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Unique teams:  {pd.concat([df['home_team'], df['away_team']]).nunique()}")
df.head()

## 2. Elo Rating System

Elo ratings are updated after every match. Weights account for tournament importance (K-factor), goal margin, and home advantage.

In [ ]:
def get_k_factor(tournament):
    t = tournament.lower()
    if 'fifa world cup' in t and 'qualification' not in t: return 60
    elif any(x in t for x in ['confederation','continental','copa america','euro','africa cup','gold cup','asian cup']): return 50
    elif any(x in t for x in ['qualification','qualifier']): return 40
    elif 'friendly' in t: return 20
    else: return 35

def goal_diff_multiplier(gd):
    if gd == 1: return 1.0
    elif gd == 2: return 1.5
    elif gd == 3: return 1.75
    else: return 1.75 + (gd - 3) * 0.05

elo_ratings = {}
match_elos  = []

for _, row in df.iterrows():
    h, a   = row['home_team'], row['away_team']
    if h not in elo_ratings: elo_ratings[h] = 1500
    if a not in elo_ratings: elo_ratings[a] = 1500
    he, ae = elo_ratings[h], elo_ratings[a]
    ha     = 0 if row['neutral'] else 100
    exp_h  = 1 / (1 + 10**(-(he + ha - ae) / 400))
    actual = 1.0 if row['home_score'] > row['away_score'] else (0.5 if row['home_score'] == row['away_score'] else 0.0)
    gd     = abs(row['home_score'] - row['away_score'])
    delta  = get_k_factor(row['tournament']) * goal_diff_multiplier(gd) * (actual - exp_h)
    elo_ratings[h] += delta
    elo_ratings[a] -= delta
    match_elos.append({'date': row['date'], 'neutral': row['neutral'],
                       'elo_diff': he - ae, 'result': actual})

match_df = pd.DataFrame(match_elos)
print(f"Matches processed: {len(match_df):,}")
print("\nTop 20 teams by current Elo:")
pd.Series(elo_ratings).sort_values(ascending=False).head(20).round(0).astype(int)

## 3. Match Outcome Model

Logistic regression on Elo difference predicts win probability. Time-based validation (trained pre-2018, tested 2018–present) prevents look-ahead bias.

In [ ]:
mdf = match_df[(match_df['date'].dt.year >= 1970) & (match_df['result'] != 0.5)].copy()
mdf['home_win'] = (mdf['result'] == 1.0).astype(int)
mdf['home_adv'] = (~mdf['neutral']).astype(int)

split   = pd.Timestamp('2018-01-01')
X_train = mdf[mdf['date'] < split][['elo_diff', 'home_adv']]
X_test  = mdf[mdf['date'] >= split][['elo_diff', 'home_adv']]
y_train = mdf[mdf['date'] < split]['home_win']
y_test  = mdf[mdf['date'] >= split]['home_win']

lr = LogisticRegression()
lr.fit(X_train, y_train)
y_proba = lr.predict_proba(X_test)[:, 1]

print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")
print(f"ROC-AUC:     {roc_auc_score(y_test, y_proba):.3f}")
print(f"Brier Score: {brier_score_loss(y_test, y_proba):.3f}  (lower is better, 0.25 = random)")

## 4. Official 2026 World Cup Groups

Groups are fixed based on the official FIFA draw (December 5, 2025). Playoff spots resolved:
- UEFA Playoff A → Bosnia and Herzegovina
- UEFA Playoff B → Sweden  
- UEFA Playoff C → Turkey
- UEFA Playoff D → Czechia
- Intercontinental Playoff 1 → DR Congo
- Intercontinental Playoff 2 → Iraq

In [ ]:
# Some teams have different names in the historical dataset
ELO_LOOKUP = {
    'Czechia':                 elo_ratings.get('Czech Republic', 1500),
    'Bosnia and Herzegovina':  elo_ratings.get('Bosnia and Herzegovina',
                                elo_ratings.get('Bosnia-Herzegovina', 1500)),
    'Turkey':                  elo_ratings.get('Turkey', 1500),
    'Ivory Coast':             elo_ratings.get('Ivory Coast',
                                elo_ratings.get("Côte d'Ivoire", 1500)),
    'DR Congo':                elo_ratings.get('DR Congo', 1500),
    'Curacao':                 elo_ratings.get('Curaçao', 1500),
}

def get_elo(team):
    return ELO_LOOKUP.get(team, elo_ratings.get(team, 1500))

GROUPS = {
    'A': ['Mexico',        'South Africa',          'South Korea',  'Czechia'],
    'B': ['Canada',        'Bosnia and Herzegovina','Qatar',         'Switzerland'],
    'C': ['Brazil',        'Morocco',               'Haiti',         'Scotland'],
    'D': ['United States', 'Paraguay',              'Australia',     'Turkey'],
    'E': ['Germany',       'Curacao',               'Ivory Coast',   'Ecuador'],
    'F': ['Netherlands',   'Japan',                 'Sweden',        'Tunisia'],
    'G': ['Belgium',       'Egypt',                 'Iran',          'New Zealand'],
    'H': ['Spain',         'Cape Verde',            'Saudi Arabia',  'Uruguay'],
    'I': ['France',        'Senegal',               'Iraq',          'Norway'],
    'J': ['Argentina',     'Algeria',               'Austria',       'Jordan'],
    'K': ['Portugal',      'DR Congo',              'Uzbekistan',    'Colombia'],
    'L': ['England',       'Croatia',               'Ghana',         'Panama'],
}

all_teams = [t for g in GROUPS.values() for t in g]
team_elos = {t: get_elo(t) for t in all_teams}

print(f"{'Group':<8} {'Team':<28} {'Elo':>6}")
print('-' * 45)
for gname, teams in GROUPS.items():
    for t in teams:
        print(f"  {gname:<6} {t:<28} {int(team_elos[t]):>6}")
    print()

In [ ]:
# Visualize group strength by average Elo
group_avg_elo = {g: np.mean([team_elos[t] for t in teams]) for g, teams in GROUPS.items()}

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, (gname, teams) in enumerate(GROUPS.items()):
    elos  = [team_elos[t] for t in teams]
    colors = ['gold' if e == max(elos) else 'steelblue' for e in elos]
    axes[i].barh(teams[::-1], elos[::-1], color=colors[::-1], edgecolor='white')
    axes[i].set_title(f'Group {gname}', fontsize=11, fontweight='bold')
    axes[i].set_xlim(1400, 2200)
    axes[i].tick_params(axis='both', labelsize=8)

plt.suptitle('2026 FIFA World Cup — Group Strength by Elo Rating', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 5. Group Stage Simulation

Each group is simulated with win/draw/loss probabilities derived from Elo difference. Top 2 per group (24 teams) + 8 best third-place teams advance to Round of 32.

In [ ]:
def wdl_probs(elo_a, elo_b):
    """Win/Draw/Loss probabilities on neutral ground."""
    d      = elo_a - elo_b
    p_draw = 0.25 * np.exp(-abs(d) / 500)
    p_win  = (1 / (1 + 10**(-d / 400))) * (1 - p_draw)
    p_loss = 1 - p_win - p_draw
    return p_win, p_draw, p_loss

def sim_group_match(a, b):
    p_win, p_draw, _ = wdl_probs(team_elos[a], team_elos[b])
    r = np.random.random()
    if r < p_win:              return 3, 0
    elif r < p_win + p_draw:   return 1, 1
    else:                      return 0, 3

def sim_group(group):
    pts = {t: 0 for t in group}
    for i in range(len(group)):
        for j in range(i+1, len(group)):
            pa, pb = sim_group_match(group[i], group[j])
            pts[group[i]] += pa
            pts[group[j]] += pb
    ranked = sorted(group, key=lambda t: (-pts[t], -team_elos[t]))
    return ranked, pts

# Show one example group stage
print("Example group stage results (one simulation run):\n")
for gname, group in GROUPS.items():
    ranked, pts = sim_group(list(group))
    print(f"  Group {gname}")
    for rank, team in enumerate(ranked, 1):
        q = 'Q' if rank <= 2 else ' '
        print(f"    {q} {rank}. {team:<28} {pts[team]} pts")
    print()

## 6. Full Tournament Simulation — 10,000 Runs

**Knockout format:** Round of 32 → Round of 16 → Quarter-finals → Semi-finals → 3rd Place match + Final.

In [ ]:
def sim_knockout_match(a, b):
    d = team_elos[a] - team_elos[b]
    p = lr.predict_proba([[d, 0]])[0][1]
    return (a, b) if np.random.random() < p else (b, a)

def sim_knockout_stage(teams_32):
    r = teams_32[:]
    np.random.shuffle(r)
    while len(r) > 4:
        r = [sim_knockout_match(r[i], r[i+1])[0] for i in range(0, len(r), 2)]
    sf_w1, sf_l1 = sim_knockout_match(r[0], r[1])
    sf_w2, sf_l2 = sim_knockout_match(r[2], r[3])
    champion, runner_up = sim_knockout_match(sf_w1, sf_w2)
    third,    fourth    = sim_knockout_match(sf_l1, sf_l2)
    return champion, runner_up, third, fourth

def sim_full_tournament():
    qualifiers  = []
    third_place = []
    for group in GROUPS.values():
        ranked, pts = sim_group(list(group))
        qualifiers.append(ranked[0])
        qualifiers.append(ranked[1])
        third_place.append((ranked[2], pts[ranked[2]]))
    best_third = sorted(third_place, key=lambda x: (-x[1], -team_elos[x[0]]))[:8]
    qualifiers += [t[0] for t in best_third]
    return sim_knockout_stage(qualifiers)

N_SIMS    = 10_000
positions = {t: {1:0, 2:0, 3:0, 4:0} for t in all_teams}

for _ in range(N_SIMS):
    p1, p2, p3, p4 = sim_full_tournament()
    positions[p1][1] += 1
    positions[p2][2] += 1
    positions[p3][3] += 1
    positions[p4][4] += 1

print(f"{N_SIMS:,} simulations complete.")

## 7. Results — Final Standings Probabilities

In [ ]:
rows = [{'Team': t, 'Group': next(g for g, ts in GROUPS.items() if t in ts),
         'Elo': int(team_elos[t]),
         'Champion':  positions[t][1] / N_SIMS,
         'Runner-up': positions[t][2] / N_SIMS,
         '3rd Place': positions[t][3] / N_SIMS,
         '4th Place': positions[t][4] / N_SIMS} for t in all_teams]

results = pd.DataFrame(rows).sort_values('Champion', ascending=False).reset_index(drop=True)

display_df = results.copy()
for col in ['Champion', 'Runner-up', '3rd Place', '4th Place']:
    display_df[col] = display_df[col].map(lambda x: f'{x:.1%}')

print("2026 World Cup Final Standings Probabilities")
display_df

In [ ]:
# Grouped bar chart — top 12 contenders
top12  = results.head(12)
cols   = ['Champion', 'Runner-up', '3rd Place', '4th Place']
colors = ['gold', 'silver', '#cd7f32', '#6baed6']

fig, ax = plt.subplots(figsize=(13, 7))
w = 0.18
x = np.arange(len(top12))

for i, (col, color) in enumerate(zip(cols, colors)):
    ax.bar(x + i*w, top12[col], w, label=col, color=color, edgecolor='white')

ax.set_xticks(x + w*1.5)
ax.set_xticklabels(top12['Team'], rotation=30, ha='right')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
ax.set_title('2026 FIFA World Cup — Final Standings Probabilities (Top 12)', fontsize=14)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Champion probability — all teams with >0.1% chance
champ = results[results['Champion'] > 0.001].set_index('Team')['Champion']

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(champ.index[::-1], champ.values[::-1], color='steelblue', edgecolor='white')
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
ax.set_title('2026 FIFA World Cup — Champion Probability', fontsize=14)
for i, v in enumerate(champ.values[::-1]):
    ax.text(v + 0.001, i, f'{v:.1%}', va='center', fontsize=8)
plt.tight_layout()
plt.show()

## 8. Head-to-Head Predictor

In [ ]:
def predict_match(team_a, team_b):
    elo_a    = get_elo(team_a)
    elo_b    = get_elo(team_b)
    prob_a   = lr.predict_proba([[elo_a - elo_b, 0]])[0][1]
    print(f"{team_a} ({int(elo_a)}) vs {team_b} ({int(elo_b)})")
    print(f"  {team_a} wins: {prob_a:.1%}  |  {team_b} wins: {1-prob_a:.1%}")

# Group stage rivalries
predict_match('Brazil',    'Morocco')       # Group C
predict_match('Turkey',    'Australia')     # Group D
predict_match('Colombia',  'Portugal')      # Group K
predict_match('Spain',     'Uruguay')       # Group H
predict_match('France',    'Norway')        # Group I
predict_match('England',   'Croatia')       # Group L
print()
# Potential finals
predict_match('Spain',     'Argentina')
predict_match('France',    'England')

---

## Summary

**Model performance (tested on 2018–present):**
- ROC-AUC: **0.853**
- Brier Score: **0.150**

**2026 World Cup Final Standings Probabilities (10,000 simulations, official groups):**

| Team | Champion | Runner-up | 3rd Place | 4th Place |
|------|----------|-----------|-----------|----------|
| Spain | 28.0% | 10.4% | 10.1% | 2.9% |
| Argentina | 20.5% | 11.0% | 9.9% | 3.5% |
| France | 11.3% | 9.7% | 8.4% | 4.3% |
| England | 6.2% | 7.6% | 6.8% | 4.5% |
| Colombia | 4.1% | 5.5% | 5.9% | 4.6% |
| Brazil | 3.9% | 5.6% | 5.5% | 4.5% |
| Ecuador | 3.5% | 4.6% | 4.8% | 4.8% |
| Portugal | 3.5% | 4.9% | 4.9% | 4.4% |

**Notable group stage observations:**
- **Group C** (Brazil, Morocco, Scotland, Haiti) and **Group J** (Argentina, Austria, Algeria, Jordan) are the least competitive — both favourites should advance comfortably
- **Group D** (Turkey, Paraguay, Australia, USA) is the toughest for a host nation — the USA (Elo 1795) is the weakest team in their own group
- **Group K** (Colombia, Portugal, Uzbekistan, DR Congo) is a genuine 2-horse race for the top spot

**Limitations:**
- Does not account for injuries, squad depth, or current form beyond Elo
- Knockout bracket draw is randomised in simulation; actual seeding will shift probabilities
- Draws excluded from knockout model (extra time / penalties treated as Elo-weighted coin flip)